# Análisis topológico y construcción de árboles generadores en la red vial

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import fiona

mpl.rc("text", usetex=True)
mpl.rc("font", family="serif")

In [49]:
red_cluster = gpd.read_file("/Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/asignacion_atributos/resultados_asignacion_atributos/red_clustering/red_clustering.shp", engine="fiona")

### Preparación de la red para el análisis topológico

El análisis topológico lo realizaremos sobre los links con atributos válidos y que participaron en el clustering.

In [105]:
red_topologia = red_cluster[red_cluster["cluster"] > 0].copy()
red_topologia["highway_topo"] = red_topologia["HIGHWAY"].astype("string").str.lower().str.strip()
red_topologia["es_local"] = red_topologia["highway_topo"].isin(["residential", "living_street", "service"])

In [106]:
print(f"Links de la red completa: {len(red_cluster):,}")
print(f"Links para análisis topológico: {len(red_topologia):,}")
print(f"Links residential o living_street: {red_topologia['es_local'].sum():,}")
print(f"U nulos: {red_topologia['U'].isna().sum():,}")
print(f"V nulos: {red_topologia['V'].isna().sum():,}")
print(f"KEY nulos: {red_topologia['KEY'].isna().sum():,}")
print()
print(red_topologia.loc[red_topologia["es_local"], "highway_topo"].value_counts().to_string())

Links de la red completa: 593,968
Links para análisis topológico: 456,616
Links residential o living_street: 392,147
U nulos: 0
V nulos: 0
KEY nulos: 0

highway_topo
residential      269292
service           73221
living_street     49634


## Construcción del grafo topológico

La red puede tener dos links dirigidos en los mismos nodos, uno para cada sentido. Para el análisis morfológico, ambos sentidos deben interpretarse como una misma conexión.

Construiremos un grafo no dirigido basado en pares únicos de nodos, para calcular conectividad, grados, cruces, terminales y nodos intermedios.

In [107]:
import networkx as nx

In [108]:
red_topologia["u_topo"] = red_topologia[["U", "V"]].min(axis=1).astype("int64")
red_topologia["v_topo"] = red_topologia[["U", "V"]].max(axis=1).astype("int64")

conexiones_topologicas = red_topologia[["u_topo", "v_topo"]].drop_duplicates().copy()

grafo_topologico = nx.Graph()
grafo_topologico.add_edges_from(conexiones_topologicas[["u_topo", "v_topo"]].itertuples(index=False, name=None))

grado_nodos = dict(grafo_topologico.degree())

In [109]:
serie_grados = pd.Series(grado_nodos, name="grado")

print(f"Links dirigidos analizados: {len(red_topologia):,}")
print(f"Conexiones topológicas únicas: {len(conexiones_topologicas):,}")
print(f"Nodos del grafo: {grafo_topologico.number_of_nodes():,}")
print(f"Componentes conectadas: {nx.number_connected_components(grafo_topologico):,}")
print(f"Nodos terminales, grado 1: {(serie_grados == 1).sum():,}")
print(f"Nodos intermedios, grado 2: {(serie_grados == 2).sum():,}")
print(f"Nodos de cruce, grado 3 o más: {(serie_grados >= 3).sum():,}")

Links dirigidos analizados: 456,616
Conexiones topológicas únicas: 265,089
Nodos del grafo: 196,094
Componentes conectadas: 75
Nodos terminales, grado 1: 36,694
Nodos intermedios, grado 2: 17,069
Nodos de cruce, grado 3 o más: 142,331


Con esto vemos que hay 456,616 links dirigidos, pero solo 265,089 conexiones. La diferencia de 191,527 proviene principalmente de pares en sentidos opuestos, no de calles que son redundantes. Solo hay 17,069 nodos de grado 2, y que en el mejor de nuestros casos, nos permitiría eliminar todos esos nodos intermedios, reduciendo aproximadamente 17,069 conexiones, de 265,089 a cerca de 248,020 (en la práctica será menos, porque no todos los nodos de grado 2 pueden concatenarse: algunos tienen cambios de categoría, cluster, atributos o sentido).

Por tanto, la concatenación por sí sola no nos reducirá la red dirigida de 456,616 a 300,000 links. Servirá para limpiar la segmentación, pero también vamos a necesitar una clasificación morfológica y una poda de redes locales.

### Nodos intermedios concatenables

Los nodos de grado dos son candidatos para simplificar la red porque conectan exactamente dos calles. Sin embargo, no siempre representa una segmentación geométrica apropiada.

Para considerarlo concatenable, las dos conexiones adyacentes deberán pertenecer a la red local, compartir el mismo tipo vial y conservar la misma clasificación funcional. Los nodos donde cambien estos atributos se mantendrán como puntos de separación.

In [110]:
conexiones_atributos = red_topologia.groupby(["u_topo", "v_topo"]).agg(links_originales=("KEY", "size"),
    highway_topo=("highway_topo", lambda valores: tuple(sorted(set(valores.dropna())))),
    clusters=("cluster", lambda valores: tuple(sorted(set(valores.dropna())))),
    subclusters=("subcluster", lambda valores: tuple(sorted(set(valores.dropna()))))).reset_index()

conexiones_atributos["es_local_fisica"] = conexiones_atributos["highway_topo"].apply(lambda valores: len(valores) == 1 and valores[0] in ["residential", "living_street", "service"])

In [111]:
conexiones_por_nodo = pd.concat([conexiones_atributos.rename(columns={"u_topo": "nodo", "v_topo": "vecino"}), conexiones_atributos.rename(columns={"v_topo": "nodo", "u_topo": "vecino"})], ignore_index=True)

nodos_grado_dos = serie_grados[serie_grados == 2].index
conexiones_grado_dos = conexiones_por_nodo[conexiones_por_nodo["nodo"].isin(nodos_grado_dos)].copy()

resumen_nodos_grado_dos = conexiones_grado_dos.groupby("nodo").agg(
    conexiones=("vecino", "size"),
    ambas_locales=("es_local_fisica", "all"),
    highway_iguales=("highway_topo", lambda valores: len(set(valores)) == 1),
    cluster_igual=("clusters", lambda valores: len(set(valores)) == 1),
    subcluster_igual=("subclusters", lambda valores: len(set(valores)) == 1))

resumen_nodos_grado_dos["candidato_concatenacion"] = resumen_nodos_grado_dos["ambas_locales"] & resumen_nodos_grado_dos["highway_iguales"] & resumen_nodos_grado_dos["cluster_igual"] & resumen_nodos_grado_dos["subcluster_igual"]

print(f"Nodos de grado 2: {len(resumen_nodos_grado_dos):,}")
print(f"Nodos de grado 2 en red local: {resumen_nodos_grado_dos['ambas_locales'].sum():,}")
print(f"Nodos candidatos a concatenación: {resumen_nodos_grado_dos['candidato_concatenacion'].sum():,}")

Nodos de grado 2: 17,069
Nodos de grado 2 en red local: 12,768
Nodos candidatos a concatenación: 11,868


En el mejor caso, cada nodo concatenado reduce aproximadamente una sola conexión. Por tanto, el orden de magnitud de la reducción sería de unos 8,173 segmentos.

## Links estructurales y estructuras locales

Para identificar una reducción adicional sin romper la conectividad, necesitamos distinguir los links estructurales de aquellos que forman parte de ramales, circuitos locales o conjuntos internos.

Primero vamos a identificar los puentes topológicos. Este es una conexión cuya eliminación aumenta el número de componentes conectadas del grafo, por lo que debe considerarse estructural y conservarse.

Después analizaremos las componentes biconexas, que permiten agrupar sectores conectados mediante múltiples rutas y distinguirlos de ramales o conjuntos unidos al resto de la ciudad por pocos accesos.

In [112]:
puentes_topologicos = {tuple(sorted(arista)) for arista in nx.bridges(grafo_topologico)}
conexiones_atributos["es_puente"] = conexiones_atributos.apply(lambda fila: (fila["u_topo"], fila["v_topo"]) in puentes_topologicos, axis=1)
print(f"Puentes topológicos: {conexiones_atributos['es_puente'].sum():,}")

Puentes topológicos: 47,651


In [113]:
resumen_puentes = conexiones_atributos.groupby("es_local_fisica").agg(conexiones=("es_puente", "size"), puentes=("es_puente", "sum"))
resumen_puentes["porcentaje_puentes"] = (resumen_puentes["puentes"] / resumen_puentes["conexiones"] * 100).round(2)
print(resumen_puentes.to_string())

                 conexiones  puentes  porcentaje_puentes
es_local_fisica                                         
False                 47933     1183                2.47
True                 217156    46468               21.40


Podemos observar que hay 180,400 conexiones físicas locales. De ellas, 31,395 son puentes y deben conservarse. Quedan 149,005 conexiones locales no puente que pueden estudiarse como red potencialmente redundante.

Lo siguiente es agrupar la red local en componentes morfológicas y contar cuántos accesos tiene cada una hacia el resto de la red.

## Componentes locales y puntos de acceso

Para caracterizar los conjuntos `residential` y `living_street` construiremos un subgrafo compuesto únicamente por conexiones locales. Después se identificaremos sus componentes conectadas y se contaremos cuántos nodos de cada componente tienen conexión con vialidades de otra jerarquía.

Con esto, podremos ver:

- conjuntos locales aislados
- conjuntos conectados al resto de la red por un único acceso
- conjuntos con dos accesos
- redes locales integradas mediante múltiples accesos

Las componentes con pocos accesos son candidatas a representar estructuras internas, como fraccionamientos, estacionamientos o pequeños conjuntos residenciales.

In [114]:
conexiones_locales = conexiones_atributos[conexiones_atributos["es_local_fisica"]].copy()
grafo_local = nx.Graph()
grafo_local.add_edges_from(conexiones_locales[["u_topo", "v_topo"]].itertuples(index=False, name=None))

componentes_locales = list(nx.connected_components(grafo_local))
componente_por_nodo = {nodo: componente for componente, nodos in enumerate(componentes_locales, start=1) for nodo in nodos}

In [115]:
nodos_conexion_no_local = set(conexiones_atributos.loc[~conexiones_atributos["es_local_fisica"], "u_topo"]) | set(conexiones_atributos.loc[~conexiones_atributos["es_local_fisica"], "v_topo"])
resumen_componentes_locales = []

for componente, nodos in enumerate(componentes_locales, start=1):
    subgrafo = grafo_local.subgraph(nodos)
    nodos_acceso = set(nodos) & nodos_conexion_no_local

    resumen_componentes_locales.append({
        "componente_local": componente,
        "nodos": subgrafo.number_of_nodes(),
        "conexiones": subgrafo.number_of_edges(),
        "accesos_red_no_local": len(nodos_acceso)
    })

resumen_componentes_locales = pd.DataFrame(resumen_componentes_locales)

In [116]:
print(f"Componentes locales: {len(resumen_componentes_locales):,}")
print(f"Componentes sin acceso a red no local: {(resumen_componentes_locales['accesos_red_no_local'] == 0).sum():,}")
print(f"Componentes con un acceso: {(resumen_componentes_locales['accesos_red_no_local'] == 1).sum():,}")
print(f"Componentes con dos accesos: {(resumen_componentes_locales['accesos_red_no_local'] == 2).sum():,}")
print(f"Componentes con tres accesos o más: {(resumen_componentes_locales['accesos_red_no_local'] >= 3).sum():,}")
print()
print(resumen_componentes_locales[["nodos", "conexiones", "accesos_red_no_local"]].describe().round(2))

Componentes locales: 9,824
Componentes sin acceso a red no local: 67
Componentes con un acceso: 6,617
Componentes con dos accesos: 1,915
Componentes con tres accesos o más: 1,225

          nodos  conexiones  accesos_red_no_local
count   9824.00     9824.00               9824.00
mean      18.63       22.10                  3.03
std      342.61      448.62                 54.44
min        1.00        1.00                  0.00
25%        2.00        1.00                  1.00
50%        2.00        1.00                  1.00
75%        6.00        5.00                  2.00
max    29309.00    38475.00               5030.00


### Evaluación del tamaño de las componentes locales

Una componente con un solo acceso puede ser un pequeño estacionamiento, pero también una colonia completa conectada por una única vialidad. Por esto mismo vamos a considerar el tamaño de cada componente, medido mediante su número de nodos, conexiones físicas y links originales.

In [117]:
conexiones_locales = conexiones_atributos.loc[conexiones_atributos["es_local_fisica"], ["u_topo", "v_topo", "es_puente"]].copy()
conexiones_locales["componente_local"] = conexiones_locales["u_topo"].map(componente_por_nodo)

links_por_conexion = red_topologia.groupby(["u_topo", "v_topo"]).size().rename("links_originales").reset_index()
conexiones_locales = conexiones_locales.merge(links_por_conexion, on=["u_topo", "v_topo"], how="left", validate="one_to_one")

resumen_links_componentes = conexiones_locales.groupby("componente_local").agg(conexiones_fisicas=("u_topo", "size"), links_originales=("links_originales", "sum")).reset_index()

resumen_componentes_locales = resumen_componentes_locales.drop(columns=["conexiones_fisicas", "links_originales"], errors="ignore")
resumen_componentes_locales = resumen_componentes_locales.merge(resumen_links_componentes, on="componente_local", how="left", validate="one_to_one")

In [118]:
escenarios_componentes = []

for max_conexiones in [5, 10, 20, 50, 100]:
    seleccion = resumen_componentes_locales["conexiones_fisicas"].le(max_conexiones) & resumen_componentes_locales["accesos_red_no_local"].le(2)
    escenarios_componentes.append({"max_conexiones": max_conexiones, "componentes": seleccion.sum(), "conexiones_fisicas": resumen_componentes_locales.loc[seleccion, "conexiones_fisicas"].sum(), "links_originales": resumen_componentes_locales.loc[seleccion, "links_originales"].sum()})
escenarios_componentes = pd.DataFrame(escenarios_componentes)

print(escenarios_componentes.to_string(index=False))

 max_conexiones  componentes  conexiones_fisicas  links_originales
              5         7441               12047             25193
             10         8023               16483             34175
             20         8372               21519             43658
             50         8551               26852             53358
            100         8582               28895             56890


Aquí vemos que las componentes locales pequeñas no contienen suficiente volumen: Con un límite de 5 conexiones solo retiraríamos potencialmente 9,317 links. Con un límite de 100 conexiones como máximo entrarían 27,618 links originales. Para pasar de 456,616 a aproximadamente 300,000, necesitaríamos reducir alrededor de 156,616 links.

Por lo tanto, aun usando el mejor escenario, las componentes pequeñas aportarían menos del 18 % de la reducción. La mayor parte de `residential` y `living_street` está dentro de componentes grandes, probablemente redes residenciales continuas que abarcan colonias completas.

El siguiente paso lógico entonces es subdividir esas componentes grandes mediante puntos de articulación y bloques biconexos, para agrupar topologías equivalentes.

**Nota:** Un bloque biconexo es una porción del grafo donde existen rutas alternativas entre sus nodos. Los puntos de articulación son los nodos que conectan esos bloques. Así podemos detectar:

- pequeños circuitos internos
- grupos residenciales conectados al resto por uno o pocos nodos
- ramificaciones dentro de componentes mucho más grandes
- estructuras locales que ahora quedaron ocultas dentro de una sola componente

## Descomposición de la red local en bloques biconexos

In [119]:
puntos_articulacion_local = set(nx.articulation_points(grafo_local))
bloques_biconexos = list(nx.biconnected_components(grafo_local))

resumen_bloques = []
for bloque, nodos in enumerate(bloques_biconexos, start=1):
    subgrafo = grafo_local.subgraph(nodos)
    nodos_articulacion = set(nodos) & puntos_articulacion_local
    nodos_acceso = set(nodos) & nodos_conexion_no_local
    resumen_bloques.append({"bloque_topologico": bloque, "nodos": subgrafo.number_of_nodes(), "conexiones_fisicas": subgrafo.number_of_edges(), "articulaciones": len(nodos_articulacion), "accesos_red_no_local": len(nodos_acceso)})

resumen_bloques = pd.DataFrame(resumen_bloques)

In [120]:
print(f"Puntos de articulación en la red local: {len(puntos_articulacion_local):,}")
print(f"Bloques biconexos: {len(resumen_bloques):,}")
print()
print(resumen_bloques[["nodos", "conexiones_fisicas", "articulaciones", "accesos_red_no_local"]].describe().round(2))
print()
print("Bloques por tamaño:")
print(pd.cut(resumen_bloques["conexiones_fisicas"], bins=[0, 1, 5, 10, 20, 50, 100, np.inf], labels=["1", "2-5", "6-10", "11-20", "21-50", "51-100", "Más de 100"]).value_counts().sort_index().to_string())

Puntos de articulación en la red local: 54,165
Bloques biconexos: 78,452

          nodos  conexiones_fisicas  articulaciones  accesos_red_no_local
count  78452.00            78452.00        78452.00              78452.00
mean       3.21                2.77            1.57                  0.42
std       63.73               93.13           16.09                  6.08
min        2.00                1.00            0.00                  0.00
25%        2.00                1.00            1.00                  0.00
50%        2.00                1.00            1.00                  0.00
75%        2.00                1.00            2.00                  1.00
max    16240.00            23780.00         4111.00               1655.00

Bloques por tamaño:
conexiones_fisicas
1             73807
2-5            2283
6-10            944
11-20           675
21-50           453
51-100          130
Más de 100      160


De los 56,931 bloques biconexos, 53,831 tienen una sola conexión. Esos bloques de tamaño 1 son ramas o puentes dentro de la red local. No representan redundancia y no las podemos poder. Solo 3,100 bloques tienen más de una conexión y contienen algún tipo de circuito o ruta alternativa.

Por lo tanto, ya no podemos eliminar bloques pequeños, sino medir la redundancia cíclica dentro de cada bloque.

Para un bloque conectado con: $E$ conexiones y $V$ nodos, el mínimo necesario para mantenerlo conectado es $V-1$, por lo tanto, la redundancia topológica es:

$$E-(V-1),$$

valor que nos indica cuántas conexiones físicas podrían retirarse, como máximo teórico, conservando conectados todos los nodos del bloque.

## Redundancia cíclica

Esta medida no define todavía cuáles conexiones deben eliminarse. Únicamente permite estimar el potencial máximo de reducción topológica.

In [121]:
resumen_bloques["redundancia_ciclica"] = resumen_bloques["conexiones_fisicas"] - resumen_bloques["nodos"] + 1
resumen_bloques["tiene_ciclo"] = resumen_bloques["redundancia_ciclica"] > 0

In [122]:
conexiones_locales_totales = grafo_local.number_of_edges()
redundancia_total = resumen_bloques["redundancia_ciclica"].sum()
conexiones_minimas = conexiones_locales_totales - redundancia_total

print(f"Conexiones físicas locales: {conexiones_locales_totales:,}")
print(f"Bloques con ciclos: {resumen_bloques['tiene_ciclo'].sum():,}")
print(f"Redundancia cíclica total: {redundancia_total:,}")
print(f"Conexiones mínimas para conservar conectividad local: {conexiones_minimas:,}")
print(f"Reducción física máxima teórica: {redundancia_total / conexiones_locales_totales * 100:.2f}%")

Conexiones físicas locales: 217,156
Bloques con ciclos: 4,645
Redundancia cíclica total: 43,944
Conexiones mínimas para conservar conectividad local: 173,212
Reducción física máxima teórica: 20.24%


In [123]:
resumen_redundancia = pd.cut(resumen_bloques.loc[resumen_bloques["tiene_ciclo"], "conexiones_fisicas"], bins=[1, 5, 10, 20, 50, 100, np.inf], 
                             labels=["2-5", "6-10", "11-20", "21-50", "51-100", "Más de 100"]).to_frame("rango")
resumen_redundancia["redundancia_ciclica"] = resumen_bloques.loc[resumen_bloques["tiene_ciclo"], "redundancia_ciclica"].values
resumen_redundancia = resumen_redundancia.groupby("rango", observed=False).agg(bloques=("rango", "size"), redundancia_ciclica=("redundancia_ciclica", "sum"))
print(resumen_redundancia.to_string())

            bloques  redundancia_ciclica
rango                                   
2-5            2283                 2415
6-10            944                 2068
11-20           675                 2876
21-50           453                 4087
51-100          130                 2748
Más de 100      160                29750


38,229 conexiones físicas son redundantes en el máximo teórico. De ellas, 27,920 (aproximadamente 73%) están en solo 148 bloques con más de 100 conexiones. Los bloques pequeños y medianos concentran únicamente 10,309 conexiones redundantes.

Esto implica que eliminar únicamente estacionamientos, pequeños fraccionamientos o circuitos locales no será suficiente. Para obtener una reducción considerable habría que intervenir también en grandes mallas residenciales, pero convertirlas completamente en árboles destruiría demasiadas rutas alternativas. Por ahora debemos clasificar cada conexión según la estructura topológica a la que pertenece y trasladar esa clasificación a los links originales.

## Clasificación morfológica de las conexiones locales

Cada conexión física local se asociará con su bloque biconexo y se clasificará de acuerdo con la escala de la estructura a la que pertenece:

- puente o ramal
- circuito local pequeño
- malla local mediana
- malla local grande

In [124]:
bloques_biconexos_aristas = list(nx.biconnected_component_edges(grafo_local))
registros_bloques = []
for bloque, aristas in enumerate(bloques_biconexos_aristas, start=1):
    for u_topo, v_topo in aristas:
        registros_bloques.append({"u_topo": min(u_topo, v_topo), "v_topo": max(u_topo, v_topo), "bloque_topologico": bloque})
bloque_por_conexion = pd.DataFrame(registros_bloques)

conexiones_locales = conexiones_locales.drop(columns=["bloque_topologico", "nodos_bloque", "conexiones_bloque", "redundancia_ciclica", "tipo_topologico"], errors="ignore")
conexiones_locales = conexiones_locales.merge(bloque_por_conexion, on=["u_topo", "v_topo"], how="left", validate="one_to_one")
atributos_bloques = resumen_bloques[["bloque_topologico", "nodos", "conexiones_fisicas", "redundancia_ciclica"]].rename(columns={"nodos": "nodos_bloque", "conexiones_fisicas": "conexiones_bloque"})
conexiones_locales = conexiones_locales.merge(atributos_bloques, on="bloque_topologico", how="left", validate="many_to_one")

In [125]:
resumen_bloques = []
for bloque, aristas in enumerate(bloques_biconexos_aristas, start=1):
    aristas_normalizadas = [(min(u, v), max(u, v)) for u, v in aristas]
    nodos = {nodo for arista in aristas_normalizadas for nodo in arista}
    nodos_articulacion = nodos & puntos_articulacion_local
    nodos_acceso = nodos & nodos_conexion_no_local
    conexiones_fisicas = len(aristas_normalizadas)
    redundancia_ciclica = conexiones_fisicas - len(nodos) + 1

    resumen_bloques.append({
        "bloque_topologico": bloque,
        "nodos": len(nodos),
        "conexiones_fisicas": conexiones_fisicas,
        "articulaciones": len(nodos_articulacion),
        "accesos_red_no_local": len(nodos_acceso),
        "redundancia_ciclica": redundancia_ciclica,
        "tiene_ciclo": redundancia_ciclica > 0
    })

resumen_bloques = pd.DataFrame(resumen_bloques)

In [126]:
atributos_bloques = resumen_bloques[["bloque_topologico", "nodos", "conexiones_fisicas", "articulaciones", "accesos_red_no_local", "redundancia_ciclica", "tiene_ciclo"]].rename(columns={
    "nodos": "nodos_bloque",
    "conexiones_fisicas": "conexiones_bloque",
    "articulaciones": "articulaciones_bloque",
    "accesos_red_no_local": "accesos_bloque"
})

conexiones_locales = conexiones_locales.drop(columns=["nodos_bloque", "conexiones_bloque", "articulaciones_bloque", "accesos_bloque", "redundancia_ciclica", "tiene_ciclo", "tipo_topologico"], errors="ignore")
conexiones_locales = conexiones_locales.merge(atributos_bloques, on="bloque_topologico", how="left", validate="many_to_one")

In [127]:
condiciones_topologicas = [
    conexiones_locales["u_topo"].eq(conexiones_locales["v_topo"]),
    conexiones_locales["redundancia_ciclica"].eq(0),
    conexiones_locales["conexiones_bloque"].le(20),
    conexiones_locales["conexiones_bloque"].le(100),
    conexiones_locales["conexiones_bloque"].gt(100)
]

categorias_topologicas = ["autoenlace", "ramal_puente", "circuito_local", "malla_local_media", "malla_local_grande"]
conexiones_locales["tipo_topologico"] = np.select(condiciones_topologicas, categorias_topologicas, default="sin_clasificar")

In [128]:
resumen_tipo_topologico = conexiones_locales.groupby("tipo_topologico").agg(conexiones_fisicas=("u_topo", "size"), links_originales=("links_originales", "sum"), bloques=("bloque_topologico", "nunique")).reset_index()

print(resumen_tipo_topologico.to_string(index=False))
print()
print(f"Conexiones clasificadas: {(conexiones_locales['tipo_topologico'] != 'sin_clasificar').sum():,}")
print(f"Conexiones sin clasificar: {(conexiones_locales['tipo_topologico'] == 'sin_clasificar').sum():,}")

   tipo_topologico  conexiones_fisicas  links_originales  bloques
        autoenlace                 760              4299      608
    circuito_local               23212             43220     3736
malla_local_grande               96314            161052      160
 malla_local_media               22897             40182      583
      ramal_puente               73973            141635    73973

Conexiones clasificadas: 217,156
Conexiones sin clasificar: 0


## Separación de autoenlaces

La red contiene conexiones cuyo nodo inicial y final son iguales. Estas conexiones pueden representar glorietas, circuitos cerrados o geometrías especiales, pero no conectan dos nodos diferentes.

Por esta razón, los autoenlaces se conservarán como una categoría independiente y no participarán en el cálculo de puentes, puntos de articulación, componentes biconexas ni redundancia cíclica.

El análisis topológico se reconstruirá únicamente con conexiones entre nodos distintos.

In [129]:
conexiones_locales = conexiones_atributos[conexiones_atributos["es_local_fisica"]].copy()
conexiones_autoenlace = conexiones_locales[conexiones_locales["u_topo"] == conexiones_locales["v_topo"]].copy()
conexiones_locales_sin_autoenlace = conexiones_locales[conexiones_locales["u_topo"] != conexiones_locales["v_topo"]].copy()

grafo_local = nx.Graph()
grafo_local.add_edges_from(conexiones_locales_sin_autoenlace[["u_topo", "v_topo"]].itertuples(index=False, name=None))

In [130]:
componentes_locales = list(nx.connected_components(grafo_local))
componente_por_nodo = {nodo: componente for componente, nodos in enumerate(componentes_locales, start=1) for nodo in nodos}

puntos_articulacion_local = set(nx.articulation_points(grafo_local))
bloques_biconexos_aristas = list(nx.biconnected_component_edges(grafo_local))

In [131]:
print(f"Conexiones locales totales: {len(conexiones_locales):,}")
print(f"Autoenlaces: {len(conexiones_autoenlace):,}")
print(f"Conexiones locales sin autoenlaces: {len(conexiones_locales_sin_autoenlace):,}")
print(f"Aristas del grafo local: {grafo_local.number_of_edges():,}")

Conexiones locales totales: 217,156
Autoenlaces: 760
Conexiones locales sin autoenlaces: 216,396
Aristas del grafo local: 216,396


### Reconstrucción de bloques topológicos sin autoenlaces

Una vez separados los autoenlaces, se reconstruyen las componentes biconexas de la red local.

Cada conexión entre nodos distintos se asociará con un único bloque topológico. Para cada bloque se calcularán su número de nodos, conexiones físicas, puntos de articulación, accesos a la red no local y redundancia cíclica.

Los autoenlaces se conservarán fuera de esta descomposición y se incorporarán posteriormente como una categoría morfológica independiente.

In [132]:
registros_bloques = []
resumen_bloques = []

for bloque, aristas in enumerate(bloques_biconexos_aristas, start=1):
    aristas_normalizadas = [(min(u, v), max(u, v)) for u, v in aristas]
    nodos = {nodo for arista in aristas_normalizadas for nodo in arista}
    conexiones_fisicas = len(aristas_normalizadas)
    redundancia_ciclica = conexiones_fisicas - len(nodos) + 1

    resumen_bloques.append({
        "bloque_topologico": bloque,
        "nodos_bloque": len(nodos),
        "conexiones_bloque": conexiones_fisicas,
        "articulaciones_bloque": len(nodos & puntos_articulacion_local),
        "accesos_bloque": len(nodos & nodos_conexion_no_local),
        "redundancia_ciclica": redundancia_ciclica
    })

    for u_topo, v_topo in aristas_normalizadas:
        registros_bloques.append({
            "u_topo": u_topo,
            "v_topo": v_topo,
            "bloque_topologico": bloque
        })

resumen_bloques = pd.DataFrame(resumen_bloques)
bloque_por_conexion = pd.DataFrame(registros_bloques)

In [133]:
links_por_conexion = red_topologia.groupby(["u_topo", "v_topo"]).size().rename("links_originales").reset_index()

conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace[["u_topo", "v_topo", "es_puente"]].copy()
conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace.merge(links_por_conexion, on=["u_topo", "v_topo"], how="left", validate="one_to_one")
conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace.merge(bloque_por_conexion, on=["u_topo", "v_topo"], how="left", validate="one_to_one")
conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace.merge(resumen_bloques, on="bloque_topologico", how="left", validate="many_to_one")

In [134]:
print(f"Conexiones locales sin autoenlaces: {len(conexiones_locales_sin_autoenlace):,}")
print(f"Conexiones asignadas a bloques: {conexiones_locales_sin_autoenlace['bloque_topologico'].notna().sum():,}")
print(f"Conexiones sin bloque: {conexiones_locales_sin_autoenlace['bloque_topologico'].isna().sum():,}")
print(f"Conexiones duplicadas: {bloque_por_conexion.duplicated(['u_topo', 'v_topo']).sum():,}")
print(f"Bloques topológicos: {len(resumen_bloques):,}")
print(f"Redundancia cíclica total corregida: {resumen_bloques['redundancia_ciclica'].sum():,}")

Conexiones locales sin autoenlaces: 216,396
Conexiones asignadas a bloques: 216,396
Conexiones sin bloque: 0
Conexiones duplicadas: 0
Bloques topológicos: 78,452
Redundancia cíclica total corregida: 43,169


In [135]:
condiciones_topologicas = [
    conexiones_locales_sin_autoenlace["redundancia_ciclica"].eq(0),
    conexiones_locales_sin_autoenlace["conexiones_bloque"].le(20),
    conexiones_locales_sin_autoenlace["conexiones_bloque"].le(100),
    conexiones_locales_sin_autoenlace["conexiones_bloque"].gt(100)
]

categorias_topologicas = [
    "ramal_puente",
    "circuito_local",
    "malla_local_media",
    "malla_local_grande"
]

conexiones_locales_sin_autoenlace["tipo_topologico"] = np.select(condiciones_topologicas, categorias_topologicas, default="sin_clasificar")

In [136]:
conexiones_autoenlace = conexiones_autoenlace[["u_topo", "v_topo"]].copy()
conexiones_autoenlace = conexiones_autoenlace.merge(links_por_conexion, on=["u_topo", "v_topo"], how="left", validate="one_to_one")

conexiones_autoenlace["bloque_topologico"] = pd.NA
conexiones_autoenlace["nodos_bloque"] = 1
conexiones_autoenlace["conexiones_bloque"] = 1
conexiones_autoenlace["articulaciones_bloque"] = 0
conexiones_autoenlace["accesos_bloque"] = 0
conexiones_autoenlace["redundancia_ciclica"] = 1
conexiones_autoenlace["tipo_topologico"] = "autoenlace"

conexiones_locales_clasificadas = pd.concat([conexiones_locales_sin_autoenlace, conexiones_autoenlace], ignore_index=True)

In [137]:
resumen_tipo_topologico = conexiones_locales_clasificadas.groupby("tipo_topologico").agg(
    conexiones_fisicas=("u_topo", "size"),
    links_originales=("links_originales", "sum"),
    bloques=("bloque_topologico", "nunique")
).reset_index()

print(resumen_tipo_topologico.to_string(index=False))
print()
print(f"Conexiones físicas clasificadas: {len(conexiones_locales_clasificadas):,}")
print(f"Conexiones sin clasificar: {(conexiones_locales_clasificadas['tipo_topologico'] == 'sin_clasificar').sum():,}")
print(f"Autoenlaces clasificados: {(conexiones_locales_clasificadas['tipo_topologico'] == 'autoenlace').sum():,}")

   tipo_topologico  conexiones_fisicas  links_originales  bloques
        autoenlace                 760              4299        0
    circuito_local               22611             42018     3135
malla_local_grande               96314            161052      160
 malla_local_media               22897             40182      583
      ramal_puente               74574            142837    74574

Conexiones físicas clasificadas: 217,156
Conexiones sin clasificar: 0
Autoenlaces clasificados: 760


## Integración de la clasificación topológica

La clasificación morfológica fue calculada sobre conexiones físicas no dirigidas. Sin embargo, la red original conserva links dirigidos independientes para cada sentido de circulación.

Ahora los atributos topológicos de cada conexión los vamos a asignar a todos los links originales que comparten el mismo par de nodos.

Las vialidades que no pertenecen a las categorías `residential` o `living_street` se identificarán como `no_local`. Ningún link será eliminado durante esta integración.

In [138]:
atributos_topologicos = conexiones_locales_clasificadas[["u_topo", "v_topo", "tipo_topologico", "bloque_topologico", "nodos_bloque", "conexiones_bloque", "articulaciones_bloque", "accesos_bloque", "redundancia_ciclica"]].copy()
columnas_topologicas = ["tipo_topologico", "bloque_topologico", "nodos_bloque", "conexiones_bloque", "articulaciones_bloque", "accesos_bloque", "redundancia_ciclica"]

red_topologia = red_topologia.drop(columns=columnas_topologicas, errors="ignore")
red_topologia = red_topologia.merge(atributos_topologicos, on=["u_topo", "v_topo"], how="left", validate="many_to_one")
red_topologia["tipo_topologico"] = red_topologia["tipo_topologico"].fillna("no_local")

In [139]:
print(f"Links antes de integrar: {456616:,}")
print(f"Links después de integrar: {len(red_topologia):,}")
print(f"Links sin tipo topológico: {red_topologia['tipo_topologico'].isna().sum():,}")
print()
print(red_topologia["tipo_topologico"].value_counts().to_string())

Links antes de integrar: 456,616
Links después de integrar: 456,616
Links sin tipo topológico: 0

tipo_topologico
malla_local_grande    161052
ramal_puente          142837
no_local               66228
circuito_local         42018
malla_local_media      40182
autoenlace              4299


### Conexiones con clasificación vial mixta

Una conexión física puede estar representada por varios links dirigidos. En algunos casos, estos links no comparten la misma categoría `HIGHWAY`. Estas conexiones se excluyeron del grafo local porque no pueden clasificarse de forma inequívoca como `residential` o `living_street`.

Antes de definir candidatos de simplificación, revisaremos estas conexiones mixtas para conservar una clasificación topológica trazable y evitar eliminar accidentalmente links asociados con vialidades de mayor jerarquía.

In [140]:
conexiones_mixtas = conexiones_atributos[
    conexiones_atributos["highway_topo"].apply(lambda valores: any(valor in ["residential", "living_street", "service"] for valor in valores)) &
    ~conexiones_atributos["es_local_fisica"]
].copy()

In [141]:
pares_mixtos = set(zip(conexiones_mixtas["u_topo"], conexiones_mixtas["v_topo"]))
es_link_mixto = red_topologia.apply(lambda fila: (fila["u_topo"], fila["v_topo"]) in pares_mixtos, axis=1)

print(f"Conexiones físicas mixtas: {len(conexiones_mixtas):,}")
print(f"Links originales en conexiones mixtas: {es_link_mixto.sum():,}")
print(f"Links locales dentro de conexiones mixtas: {(es_link_mixto & red_topologia['es_local']).sum():,}")
print()
print(conexiones_mixtas["highway_topo"].value_counts().head(20).to_string())

Conexiones físicas mixtas: 561
Links originales en conexiones mixtas: 2,302
Links locales dentro de conexiones mixtas: 1,759

highway_topo
(residential, service)          118
(secondary, service)             88
(service, tertiary)              83
(primary, service)               75
(residential, tertiary)          51
(living_street, residential)     40
(service, unclassified)          28
(residential, unclassified)      19
(service, trunk)                 16
(motorway, service)              10
(living_street, tertiary)        10
(living_street, service)          9
(residential, secondary)          7
(primary_link, service)           2
(primary, residential)            2
(service, trunk_link)             2
(secondary_link, service)         1


Ahora sigue construir un primer escenario de reducción topológica, todavía sin borrar links.

La idea será:

- conservar todos los ramal_puente
- conservar todos los links no_local
- conservar todas las conexiones mixtas
- conservar provisionalmente los autoenlace
- dentro de cada bloque con ciclos, seleccionar un árbol generador que mantenga conectados todos sus nodos
- marcar como candidatos los links físicos que quedan fuera de ese árbol

Para evitar que la selección sea arbitraria, priorizaremos conservar conexiones con mayor capacidad, más carriles y mayor velocidad.

In [142]:
atributos_por_conexion = red_topologia.groupby(["u_topo", "v_topo"]).agg(capacidad=("cap_final", "median"), carriles=("carr_final", "median"), velocidad=("vel_final", "median")).reset_index()

atributos_por_conexion["rango_capacidad"] = atributos_por_conexion["capacidad"].rank(pct=True)
atributos_por_conexion["rango_carriles"] = atributos_por_conexion["carriles"].rank(pct=True)
atributos_por_conexion["rango_velocidad"] = atributos_por_conexion["velocidad"].rank(pct=True)
atributos_por_conexion["importancia_topologica"] = 0.5 * atributos_por_conexion["rango_capacidad"] + 0.3 * atributos_por_conexion["rango_carriles"] + 0.2 * atributos_por_conexion["rango_velocidad"]

conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace.drop(columns=["capacidad", "carriles", "velocidad", "importancia_topologica"], errors="ignore")
conexiones_locales_sin_autoenlace = conexiones_locales_sin_autoenlace.merge(atributos_por_conexion[["u_topo", "v_topo", "capacidad", "carriles", "velocidad", "importancia_topologica"]], on=["u_topo", "v_topo"], how="left", validate="one_to_one")

Los pesos utilizados son:

- capacidad: 50%
- carriles: 30%
- velocidad: 20%.

Todavía pueden cambiarse después. Por ahora solo necesitamos una regla reproducible.

In [143]:
conexiones_conservar = set()

for bloque, grupo in conexiones_locales_sin_autoenlace.groupby("bloque_topologico"):
    if grupo["redundancia_ciclica"].iloc[0] == 0:
        conexiones_conservar.update(zip(grupo["u_topo"], grupo["v_topo"]))
        continue

    grafo_bloque = nx.Graph()

    for fila in grupo.itertuples(index=False):
        grafo_bloque.add_edge(fila.u_topo, fila.v_topo, weight=fila.importancia_topologica)

    arbol_bloque = nx.maximum_spanning_tree(grafo_bloque, weight="weight")
    conexiones_conservar.update((min(u, v), max(u, v)) for u, v in arbol_bloque.edges())

conexiones_locales_sin_autoenlace["conservar_esqueleto"] = conexiones_locales_sin_autoenlace.apply(lambda fila: (fila["u_topo"], fila["v_topo"]) in conexiones_conservar, axis=1)
conexiones_locales_sin_autoenlace["candidato_reduccion"] = ~conexiones_locales_sin_autoenlace["conservar_esqueleto"]

In [144]:
print(f"Conexiones físicas locales analizadas: {len(conexiones_locales_sin_autoenlace):,}")
print(f"Conexiones conservadas: {conexiones_locales_sin_autoenlace['conservar_esqueleto'].sum():,}")
print(f"Conexiones candidatas a reducción: {conexiones_locales_sin_autoenlace['candidato_reduccion'].sum():,}")
print()

resumen_candidatos = conexiones_locales_sin_autoenlace.groupby("tipo_topologico").agg(
    conexiones=("u_topo", "size"),
    candidatas=("candidato_reduccion", "sum")
)

resumen_candidatos["porcentaje_candidato"] = (resumen_candidatos["candidatas"] / resumen_candidatos["conexiones"] * 100).round(2)

print(resumen_candidatos.to_string())

Conexiones físicas locales analizadas: 216,396
Conexiones conservadas: 173,227
Conexiones candidatas a reducción: 43,169

                    conexiones  candidatas  porcentaje_candidato
tipo_topologico                                                 
circuito_local           22611        6590                 29.15
malla_local_grande       96314       29747                 30.89
malla_local_media        22897        6832                 29.84
ramal_puente             74574           0                  0.00


### Validación

In [159]:
resumen_bloques["grupo_redundancia"] = pd.cut(resumen_bloques["redundancia_ciclica"], bins=[-1, 0, 2, 5, np.inf], labels=["sin_redundancia", "baja", "media", "alta"])

resultados_validacion = []

for _, fila_bloque in resumen_bloques.iterrows():
    bloque = fila_bloque["bloque_topologico"]
    grupo = conexiones_locales_sin_autoenlace[conexiones_locales_sin_autoenlace["bloque_topologico"] == bloque]

    grafo_base = nx.Graph()
    grafo_base.add_edges_from(zip(grupo["u_topo"], grupo["v_topo"]))

    grupo_arbol = grupo[grupo["conservar_esqueleto"]]

    grafo_arbol = nx.Graph()
    grafo_arbol.add_nodes_from(grafo_base.nodes())
    grafo_arbol.add_edges_from(zip(grupo_arbol["u_topo"], grupo_arbol["v_topo"]))

    resultados_validacion.append({
        "bloque_topologico": bloque,
        "grupo_redundancia": fila_bloque["grupo_redundancia"],
        "nodos_base": grafo_base.number_of_nodes(),
        "conexiones_base": grafo_base.number_of_edges(),
        "conexiones_arbol": grafo_arbol.number_of_edges(),
        "conexiones_eliminadas": grafo_base.number_of_edges() - grafo_arbol.number_of_edges(),
        "redundancia_base": grafo_base.number_of_edges() - grafo_base.number_of_nodes() + 1,
        "mismos_nodos": set(grafo_base.nodes()) == set(grafo_arbol.nodes()),
        "base_conexa": nx.is_connected(grafo_base),
        "arbol_conexo": nx.is_connected(grafo_arbol),
        "arbol_sin_ciclos": nx.is_tree(grafo_arbol),
        "cumple_n_menos_1": grafo_arbol.number_of_edges() == grafo_arbol.number_of_nodes() - 1
    })

resultados_validacion = pd.DataFrame(resultados_validacion)

resultados_validacion["eliminacion_correcta"] = resultados_validacion["conexiones_eliminadas"] == resultados_validacion["redundancia_base"]
resultados_validacion["validacion_correcta"] = resultados_validacion[["mismos_nodos", "base_conexa", "arbol_conexo", "arbol_sin_ciclos", "cumple_n_menos_1", "eliminacion_correcta"]].all(axis=1)

columnas_booleanas = ["mismos_nodos", "base_conexa", "arbol_conexo", "arbol_sin_ciclos", "cumple_n_menos_1", "eliminacion_correcta", "validacion_correcta"]

resumen_validacion = pd.DataFrame({"todos_true": resultados_validacion[columnas_booleanas].all(), "true": resultados_validacion[columnas_booleanas].sum(), "false": (~resultados_validacion[columnas_booleanas]).sum()})
display(resumen_validacion)

print(f"Bloques evaluados: {len(resultados_validacion):,}")
print(f"Bloques correctamente reducidos: {resultados_validacion['validacion_correcta'].sum():,}")
print(f"Bloques con problemas: {(~resultados_validacion['validacion_correcta']).sum():,}")

,todos_true,true,false
mismos_nodos,True,78452,0
base_conexa,True,78452,0
arbol_conexo,True,78452,0
arbol_sin_ciclos,True,78452,0
cumple_n_menos_1,True,78452,0
eliminacion_correcta,True,78452,0
validacion_correcta,True,78452,0


Bloques evaluados: 78,452
Bloques correctamente reducidos: 78,452
Bloques con problemas: 0


### Traslado de candidatos de reducción a los links originales

Los links se clasificarán como:

- `conservar`: pertenecen al esqueleto conectado o no forman parte de la red local analizada;
- `candidato_reduccion`: pertenecen a una conexión física redundante que quedó fuera del árbol generador.

In [149]:
decision_por_conexion = conexiones_locales_sin_autoenlace[["u_topo", "v_topo", "conservar_esqueleto", "candidato_reduccion"]].copy()

In [150]:
red_topologia = red_topologia.drop(columns=["conservar_esqueleto", "candidato_reduccion"], errors="ignore")
red_topologia = red_topologia.merge(decision_por_conexion, on=["u_topo", "v_topo"], how="left", validate="many_to_one")

red_topologia["conservar_esqueleto"] = red_topologia["conservar_esqueleto"].fillna(True).astype(bool)
red_topologia["candidato_reduccion"] = red_topologia["candidato_reduccion"].fillna(False).astype(bool)

/var/folders/2j/8mgsmg092vb0t8zngc_qmw480000gn/T/ipykernel_22192/3117958952.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  red_topologia["conservar_esqueleto"] = red_topologia["conservar_esqueleto"].fillna(True).astype(bool)
/var/folders/2j/8mgsmg092vb0t8zngc_qmw480000gn/T/ipykernel_22192/3117958952.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  red_topologia["candidato_reduccion"] = red_topologia["candidato_reduccion"].fillna(False).astype(bool)


In [151]:
links_candidatos = red_topologia["candidato_reduccion"].sum()
links_conservados = (~red_topologia["candidato_reduccion"]).sum()

print(f"Links dirigidos originales: {len(red_topologia):,}")
print(f"Links candidatos a reducción: {links_candidatos:,}")
print(f"Links conservados: {links_conservados:,}")
print(f"Reducción porcentual: {links_candidatos / len(red_topologia) * 100:.2f}%")
print()

resumen_reduccion = red_topologia.groupby("tipo_topologico").agg(links=("KEY", "size"), candidatos=("candidato_reduccion", "sum"))
resumen_reduccion["porcentaje_candidato"] = (resumen_reduccion["candidatos"] / resumen_reduccion["links"] * 100).round(2)
print(resumen_reduccion.to_string())

Links dirigidos originales: 456,616
Links candidatos a reducción: 75,102
Links conservados: 381,514
Reducción porcentual: 16.45%

                     links  candidatos  porcentaje_candidato
tipo_topologico                                             
autoenlace            4299           0                  0.00
circuito_local       42018       12435                 29.59
malla_local_grande  161052       50585                 31.41
malla_local_media    40182       12082                 30.07
no_local             66228           0                  0.00
ramal_puente        142837           0                  0.00


In [152]:
print()
print(f"Red clasificable después de la reducción: {len(red_topologia) - links_candidatos:,}")
print(f"Red completa después de excluir cluster -1 y -2 y aplicar la reducción: {len(red_topologia) - links_candidatos:,}")
print(f"Red completa conservando también cluster -1 y -2: {len(red_cluster) - links_candidatos:,}")


Red clasificable después de la reducción: 381,514
Red completa después de excluir cluster -1 y -2 y aplicar la reducción: 381,514
Red completa conservando también cluster -1 y -2: 518,866


In [153]:
columnas_nuevas = [
    "u_topo",
    "v_topo",
    "tipo_topologico",
    "bloque_topologico",
    "nodos_bloque",
    "conexiones_bloque",
    "articulaciones_bloque",
    "accesos_bloque",
    "redundancia_ciclica",
    "conservar_esqueleto",
    "candidato_reduccion"
]

red_topologia_final = red_cluster.copy()
resultados_topologicos = red_topologia[["U", "V", "KEY"] + columnas_nuevas].copy()

red_topologia_final["_repeticion"] = red_topologia_final.groupby(["U", "V", "KEY"]).cumcount()
resultados_topologicos["_repeticion"] = resultados_topologicos.groupby(["U", "V", "KEY"]).cumcount()

red_topologia_final = red_topologia_final.drop(columns=columnas_nuevas, errors="ignore")
red_topologia_final = red_topologia_final.merge(resultados_topologicos, on=["U", "V", "KEY", "_repeticion"], how="left", validate="one_to_one")

red_topologia_final = red_topologia_final.drop(columns="_repeticion")

red_topologia_final["tipo_topologico"] = red_topologia_final["tipo_topologico"].fillna("fuera_analisis")
red_topologia_final["conservar_esqueleto"] = red_topologia_final["conservar_esqueleto"].fillna(True).astype(bool)
red_topologia_final["candidato_reduccion"] = red_topologia_final["candidato_reduccion"].fillna(False).astype(bool)

/var/folders/2j/8mgsmg092vb0t8zngc_qmw480000gn/T/ipykernel_22192/2772255968.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  red_topologia_final["conservar_esqueleto"] = red_topologia_final["conservar_esqueleto"].fillna(True).astype(bool)
/var/folders/2j/8mgsmg092vb0t8zngc_qmw480000gn/T/ipykernel_22192/2772255968.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  red_topologia_final["candidato_reduccion"] = red_topologia_final["candidato_reduccion"].fillna(False).astype(bool)


In [154]:
red_topologia_final.head(20)

,No,from_node,to_node,TypeNo,TSysSet,Length,U,V,KEY,HIGHWAY,...,v_topo,tipo_topologico,bloque_topologico,nodos_bloque,conexiones_bloque,articulaciones_bloque,accesos_bloque,redundancia_ciclica,conservar_esqueleto,candidato_reduccion
0,1,1,103492,0,"B,C",4.827825,267537966,7306651630,0,motorway,...,7.306652e+09,no_local,NaN,NaN,NaN,NaN,NaN,NaN,True,False
1,1,103492,1,0,None,0.000000,0,0,0,None,...,NaN,fuera_analisis,NaN,NaN,NaN,NaN,NaN,NaN,True,False
2,2,1,15708,0,"B,C",0.146029,267537966,5837556433,0,motorway_link,...,5.837556e+09,no_local,NaN,NaN,NaN,NaN,NaN,NaN,True,False
3,2,15708,1,0,None,0.000000,0,0,0,None,...,NaN,fuera_analisis,NaN,NaN,NaN,NaN,NaN,NaN,True,False
4,3,2,3,0,"B,C",0.520237,267538751,273140976,0,motorway,...,2.731410e+08,no_local,NaN,NaN,NaN,NaN,NaN,NaN,True,False
5,3,3,2,0,None,0.000000,0,0,0,None,...,NaN,fuera_analisis,NaN,NaN,NaN,NaN,NaN,NaN,True,False
6,4,2,3068,0,"B,C",0.617213,267538751,1746293763,0,motorway_link,...,1.746294e+09,no_local,NaN,NaN,NaN,NaN,NaN,NaN,True,False
7,4,3068,2,0,None,0.000000,0,0,0,None,...,NaN,fuera_analisis,NaN,NaN,NaN,NaN,NaN,NaN,True,False
8,5,3,7341,0,"B,C",0.151355,273140976,1997658287,0,motorway,...,1.997658e+09,no_local,NaN,NaN,NaN,NaN,NaN,NaN,True,False
9,5,7341,3,0,None,0.000000,0,0,0,None,...,NaN,fuera_analisis,NaN,NaN,NaN,NaN,NaN,NaN,True,False


In [155]:
red_topologia_shp = red_topologia_final.rename(columns={
    "tipo_topologico": "tipo_topo",
    "bloque_topologico": "bloq_topo",
    "nodos_bloque": "nod_bloq",
    "conexiones_bloque": "con_bloq",
    "articulaciones_bloque": "art_bloq",
    "accesos_bloque": "acc_bloq",
    "redundancia_ciclica": "red_cic",
    "conservar_esqueleto": "cons_esq",
    "candidato_reduccion": "cand_red"
}).copy()

red_topologia_shp["cons_esq"] = red_topologia_shp["cons_esq"].astype("int8")
red_topologia_shp["cand_red"] = red_topologia_shp["cand_red"].astype("int8")
red_topologia_shp["bloq_topo"] = pd.to_numeric(red_topologia_shp["bloq_topo"], errors="coerce")
red_topologia_shp["nod_bloq"] = pd.to_numeric(red_topologia_shp["nod_bloq"], errors="coerce")
red_topologia_shp["con_bloq"] = pd.to_numeric(red_topologia_shp["con_bloq"], errors="coerce")
red_topologia_shp["art_bloq"] = pd.to_numeric(red_topologia_shp["art_bloq"], errors="coerce")
red_topologia_shp["acc_bloq"] = pd.to_numeric(red_topologia_shp["acc_bloq"], errors="coerce")
red_topologia_shp["red_cic"] = pd.to_numeric(red_topologia_shp["red_cic"], errors="coerce")

In [156]:
from pathlib import Path

In [157]:
carpeta_resultados_asignacion = Path("/Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/asignacion_atributos/resultados_asignacion_atributos")
carpeta_red_clustering = carpeta_resultados_asignacion / "red_topologica"
carpeta_red_clustering.mkdir(parents=True, exist_ok=True)
ruta_salida = carpeta_red_clustering / "red_topologica.shp"
red_topologia_shp.to_file(ruta_salida, driver="ESRI Shapefile", engine="fiona")

print(f"Red exportada en: {ruta_salida}")

Red exportada en: /Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/asignacion_atributos/resultados_asignacion_atributos/red_topologica/red_topologica.shp


In [158]:
red_mapa = red_topologia_final[red_topologia_final["tipo_topologico"] != "fuera_analisis"].copy()
red_mapa = red_mapa.sort_values(["u_topo", "v_topo", "candidato_reduccion"], ascending=[True, True, False])
red_mapa = red_mapa.drop_duplicates(["u_topo", "v_topo"]).copy()

red_mapa = red_mapa.to_crs(epsg=32613)
red_mapa["geometry"] = red_mapa.geometry.simplify(tolerance=3, preserve_topology=True)
red_mapa = red_mapa.to_crs(epsg=4326)

mapa_candidatos = red_mapa[red_mapa["candidato_reduccion"]].copy()
mapa_autoenlaces = red_mapa[red_mapa["tipo_topologico"] == "autoenlace"].copy()
mapa_no_local = red_mapa[red_mapa["tipo_topologico"] == "no_local"].copy()

mapa_esqueleto = red_mapa[~red_mapa["candidato_reduccion"] & ~red_mapa["tipo_topologico"].isin(["no_local", "autoenlace"])].copy()

red_mapa["bloque_mapa"] = red_mapa["bloque_topologico"].fillna(-1).astype("int64")
red_mapa["redundancia_mapa"] = red_mapa["redundancia_ciclica"].fillna(0).astype("int64")

mapa_candidatos = red_mapa[red_mapa["candidato_reduccion"]].copy()
mapa_autoenlaces = red_mapa[red_mapa["tipo_topologico"] == "autoenlace"].copy()
mapa_no_local = red_mapa[red_mapa["tipo_topologico"] == "no_local"].copy()

mapa_esqueleto = red_mapa[~red_mapa["candidato_reduccion"] & ~red_mapa["tipo_topologico"].isin(["no_local", "autoenlace"])].copy()

import folium
from folium.plugins import Fullscreen

centro_mapa = red_mapa.geometry.union_all().centroid

mapa_topologia = folium.Map(location=[centro_mapa.y, centro_mapa.x], zoom_start=10, tiles="CartoDB positron", prefer_canvas=True)
Fullscreen(position="topright").add_to(mapa_topologia)

folium.GeoJson(
    mapa_no_local[["tipo_topologico", "geometry"]],
    name=f"Red no local ({len(mapa_no_local):,})",
    show=False,
    style_function=lambda feature: {
        "color": "#8c8c8c",
        "weight": 1,
        "opacity": 0.35
    }
).add_to(mapa_topologia)


capa_esqueleto = folium.GeoJson(
    mapa_esqueleto[["tipo_topologico", "cluster", "subcluster", "bloque_mapa", "redundancia_mapa", "geometry"]],
    name=f"Esqueleto local conservado ({len(mapa_esqueleto):,})",
    show=True,
    style_function=lambda feature: {
        "color": "#2166ac",
        "weight": 1.2,
        "opacity": 0.65
    },
    highlight_function=lambda feature: {
        "color": "#000000",
        "weight": 3,
        "opacity": 1
    }
).add_to(mapa_topologia)

folium.GeoJsonTooltip(
    fields=["tipo_topologico", "cluster", "subcluster", "bloque_mapa", "redundancia_mapa"],
    aliases=["Tipo topológico:", "Cluster:", "Subcluster:", "Bloque:", "Redundancia cíclica:"],
    sticky=False
).add_to(capa_esqueleto)

capa_candidatos = folium.GeoJson(
    mapa_candidatos[["tipo_topologico", "cluster", "subcluster", "bloque_mapa", "redundancia_mapa", "geometry"]],
    name=f"Candidatos a reducción ({len(mapa_candidatos):,})",
    show=True,
    style_function=lambda feature: {
        "color": "#d73027",
        "weight": 2,
        "opacity": 0.9
    },
    highlight_function=lambda feature: {
        "color": "#000000",
        "weight": 4,
        "opacity": 1
    }
).add_to(mapa_topologia)

folium.GeoJsonTooltip(
    fields=["tipo_topologico", "cluster", "subcluster", "bloque_mapa", "redundancia_mapa"],
    aliases=["Tipo topológico:", "Cluster:", "Subcluster:", "Bloque:", "Redundancia cíclica:"],
    sticky=False
).add_to(capa_candidatos)

capa_autoenlaces = folium.GeoJson(
    mapa_autoenlaces[["tipo_topologico", "cluster", "subcluster", "geometry"]],
    name=f"Autoenlaces ({len(mapa_autoenlaces):,})",
    show=True,
    style_function=lambda feature: {
        "color": "#762a83",
        "weight": 3,
        "opacity": 0.9
    }
).add_to(mapa_topologia)

folium.GeoJsonTooltip(
    fields=["tipo_topologico", "cluster", "subcluster"],
    aliases=["Tipo topológico:", "Cluster:", "Subcluster:"],
    sticky=False
).add_to(capa_autoenlaces)

folium.LayerControl(collapsed=False).add_to(mapa_topologia)

ruta_html = carpeta_resultados_asignacion / "red_topologia.html"

mapa_topologia.save(ruta_html)

print(f"HTML exportado en: {ruta_html}")

/var/folders/2j/8mgsmg092vb0t8zngc_qmw480000gn/T/ipykernel_22192/3696946516.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  red_mapa["bloque_mapa"] = red_mapa["bloque_topologico"].fillna(-1).astype("int64")


HTML exportado en: /Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/asignacion_atributos/resultados_asignacion_atributos/red_topologia.html
